In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import ttest_ind


# Conect to Google Drive
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Portfolio/

Mounted at /content/drive
/content/drive/MyDrive/Portfolio


In [ ]:
# Uploading prepared data set
df = pd.read_csv("AB_test_portfolio.csv")
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,Ecuador,desktop,Americas,Organic Search,2,2,session with orders,1
1,2020-11-02,Trinidad & Tobago,desktop,Americas,Organic Search,2,2,session with orders,1
2,2020-11-03,Cambodia,desktop,Asia,Direct,2,1,session with orders,1
3,2020-11-03,Paraguay,desktop,Americas,Direct,2,2,session with orders,1
4,2020-11-04,Kazakhstan,desktop,Asia,Undefined,2,1,session with orders,1


In [ ]:
print(df['event_name'].unique())

['session with orders' 'session' 'new account' 'page_view'
 'user_engagement' 'first_visit' 'scroll' 'view_promotion' 'session_start'
 'view_item' 'add_to_cart' 'begin_checkout' 'select_item'
 'view_search_results' 'add_payment_info' 'select_promotion'
 'add_shipping_info' 'click' 'view_item_list']


In [ ]:
# Creating a table with key events
key_events = ['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new account']
event_denominator = 'session'

df_metrics = df[df['event_name'].isin(key_events + [event_denominator])].copy()

df_metrics.head(15)

,date,country,device,continent,channel,test,test_group,event_name,value
223,2020-11-01,Lithuania,mobile,Europe,Paid Search,2,2,session,1
224,2020-11-01,Uruguay,mobile,Americas,Undefined,2,1,session,1
225,2020-11-01,Bosnia & Herzegovina,desktop,Europe,Paid Search,2,1,session,1
226,2020-11-01,Slovenia,desktop,Europe,Organic Search,2,2,session,1
227,2020-11-01,Panama,mobile,Americas,Organic Search,2,2,session,1
228,2020-11-01,New Zealand,desktop,Oceania,Organic Search,2,1,session,1
229,2020-11-01,Slovakia,mobile,Europe,Paid Search,2,2,session,2
230,2020-11-01,Latvia,mobile,Europe,Direct,2,1,session,1
231,2020-11-01,Venezuela,desktop,Americas,Organic Search,2,2,session,1
232,2020-11-02,Puerto Rico,mobile,Americas,Direct,2,1,session,1


In [ ]:
# A/b test by key metrics

def ab_test_for_key_metrics (df):
    results = []

    test_numbers = df['test'].unique()

    for test_numb in test_numbers:
      df_test = df[df['test'] == test_numb]

      event_in_df = [e for e in df_test['event_name'].unique() if e in key_events]

      for event in event_in_df:
        group_a = df_test[df_test['test_group'] == 1]
        group_b = df_test[df_test['test_group'] == 2]

        event_a = group_a[group_a['event_name'] == event]['value'].sum()
        denominator_a = group_a[group_a['event_name'] == event_denominator]['value'].sum()

        event_b = group_b[group_b['event_name'] == event]['value'].sum()
        denominator_b = group_b[group_b['event_name'] == event_denominator]['value'].sum()

        conversion_a = event_a / denominator_a if denominator_a > 0 else 0
        conversion_b = event_b / denominator_b if denominator_b > 0 else 0

        metric_change = ((conversion_b - conversion_a) / conversion_a * 100 if conversion_a > 0 else 0)

      # z-test
        if denominator_b > 30 and denominator_b > 30:
         z_stat, p_value = sm.stats.proportions_ztest(
              [event_a, event_b],
              [denominator_a, denominator_b]
         )
         significant = p_value < 0.05
        else:
          z_stat, p_value, significant = None, None, False

        results.append({
          "test_number": test_numb,
          "metric": f"{event}/{event_denominator}",
          "numerator": event,
          "denominator": event_denominator,
          "numerator_control": event_a,
          "denominator_control": denominator_a,
          "conversion_rate_control": round(conversion_a, 3),
          "numerator_test": event_b,
          "denominator_test": denominator_b,
          "conversion_rate_test": round(conversion_b, 3),
          "metric_change_%": round(metric_change, 3),
          "z_stat": z_stat,
          "p_value": p_value,
          "significant": significant
        })

    return pd.DataFrame(results)

ab_test_key_metrics = ab_test_for_key_metrics(df_metrics)

display(ab_test_key_metrics)


,test_number,metric,numerator,denominator,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_%,z_stat,p_value,significant
0,2,new account/session,new account,session,4165,50637,0.082,4184,50244,0.083,1.242,-0.588793,0.556000,False
1,2,begin_checkout/session,begin_checkout,session,4262,50637,0.084,4313,50244,0.086,1.988,-0.952898,0.340642,False
2,2,add_payment_info/session,add_payment_info,session,2344,50637,0.046,2409,50244,0.048,3.577,-1.240994,0.214608,False
3,2,add_shipping_info/session,add_shipping_info,session,3480,50637,0.069,3510,50244,0.070,1.651,-0.709557,0.477979,False
4,1,new account/session,new account,session,3823,45362,0.084,3681,45193,0.081,-3.354,1.542883,0.122859,False
5,1,begin_checkout/session,begin_checkout,session,3784,45362,0.083,4021,45193,0.089,6.661,-2.978783,0.002894,True
6,1,add_shipping_info/session,add_shipping_info,session,3034,45362,0.067,3221,45193,0.071,6.560,-2.603571,0.009226,True
7,1,add_payment_info/session,add_payment_info,session,1988,45362,0.044,2229,45193,0.049,12.542,-3.924884,0.000087,True
8,4,new account/session,new account,session,8984,105079,0.085,8687,105141,0.083,-3.363,2.375457,0.017527,True
9,4,begin_checkout/session,begin_checkout,session,12555,105079,0.119,12267,105141,0.117,-2.352,1.995998,0.045934,True


In [ ]:
# Saving results
path = "/content/drive/MyDrive/Portfolio/ab_test_result.csv"
ab_test_key_metrics.to_csv(path)

In [ ]:
# A/b test segmentation by device and continent

def ab_test_segmentation (df):
    results = []

    test_numbers = df['test'].unique()

    for test_numb in test_numbers:
        df_test = df[df['test'] == test_numb]

        devices = df_test['device'].dropna().unique()

        for device in devices:
            df_device = df_test[df_test['device'] == device]

            continents = (df_device[df_device['continent'] != '(not set)']['continent'].dropna().unique())

            for continent in continents:
                df_segment = df_device[df_device['continent'] == continent]
                event_in_df = [e for e in df_segment['event_name'].unique() if e in key_events]

                for event in event_in_df:
                    group_a = df_segment[df_segment['test_group'] == 1]
                    group_b = df_segment[df_segment['test_group'] == 2]

                    event_a = group_a[group_a['event_name'] == event]['value'].sum()
                    denominator_a = group_a[group_a['event_name'] == event_denominator]['value'].sum()

                    event_b = group_b[group_b['event_name'] == event]['value'].sum()
                    denominator_b = group_b[group_b['event_name'] == event_denominator]['value'].sum()

                    conversion_a = event_a / denominator_a if denominator_a > 0 else 0
                    conversion_b = event_b / denominator_b if denominator_b > 0 else 0

                    metric_change = ((conversion_b - conversion_a) / conversion_a * 100 if conversion_a > 0 else 0)

                  # z-test
                    if denominator_b > 30 and denominator_b > 30:
                        z_stat, p_value = sm.stats.proportions_ztest(
                              [event_a, event_b],
                              [denominator_a, denominator_b]
                        )
                        significant = p_value < 0.05
                    else:
                      z_stat, p_value, significant = None, None, False

                    results.append({
                      "test_number": test_numb,
                      "metric": f"{event}/{event_denominator}",
                      "numerator": event,
                      "denominator": event_denominator,
                      "numerator_control": event_a,
                      "denominator_control": denominator_a,
                      "conversion_rate_control": round(conversion_a, 3),
                      "numerator_test": event_b,
                      "denominator_test": denominator_b,
                      "conversion_rate_test": round(conversion_b, 3),
                      "metric_change_%": round(metric_change, 3),
                      "z_stat": z_stat,
                      "p_value": p_value,
                      "significant": significant
                    })

    return pd.DataFrame(results)

ab_test = ab_test_segmentation(df_metrics)

display(ab_test.head())


,test_number,metric,numerator,denominator,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_%,z_stat,p_value,significant
0,2,new account/session,new account,session,285,3780,0.075,324,3664,0.088,17.283,-2.050835,0.040283,True
1,2,add_payment_info/session,add_payment_info,session,216,3780,0.057,144,3664,0.039,-31.223,3.587309,0.000334,True
2,2,add_shipping_info/session,add_shipping_info,session,277,3780,0.073,220,3664,0.060,-18.063,2.287344,0.022176,True
3,2,begin_checkout/session,begin_checkout,session,351,3780,0.093,255,3664,0.070,-25.050,3.669067,0.000243,True
4,2,new account/session,new account,session,899,10950,0.082,942,10996,0.086,4.345,-0.953089,0.340545,False


In [ ]:
path = "/content/drive/MyDrive/Portfolio/ab_test_result_2.csv"
ab_test.to_csv(path)

# Links To Dashboards

General Dashboard: https://public.tableau.com/views/ABtest_17782375681820/ABtest?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link

A/B Test Total: https://public.tableau.com/views/ABtesttotals/Dashboard1?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link

# Links To Files

A/B Test Total: https://drive.google.com/file/d/1c5rc3TpzbtnNjoIE9ImKrbghWCZ1JmPN/view?usp=sharing

A/B Test By Segments: https://drive.google.com/file/d/1nn3LZHImIdbwqURtjnf7SjqpFCOSAHXx/view?usp=sharing